# 생성기 v1: phrase graph 위 유형열 + 마스크형 이산 확산 + 그래프 손실

2026-11-11–12에 정한 설계를 구현한 실험입니다.

| 층 | 방법 |
|---|---|
| 구조 | 원곡의 phrase graph G 위에서 1차 Markov 보행으로 새 유형열 Y′를 뽑음. 시작·끝 유형 고정, 제1장(1–4마디)과 제4장(25–28마디)은 같은 유형 |
| 선율 | 마디를 격자(박당 4칸)로 적고, 유형을 조건으로 마스크형 이산 확산으로 생성. 제1장과 제4장은 한 변수로 묶어 공동 디코딩 |
| 손실 | 확산 ELBO(원곡 72마디) + λ · 그래프 손실 (i)(새 유형열을 조건으로 모두 가린 입력에서 계산) |

두 학습 조건을 비교합니다.

| 조건 | λ |
|---|---|
| `diffusion` | 0 (확산 손실만) |
| `diffusion_graph` | 1 |

학습 조건마다 학습 Seed 3개 × 보류 묶음 3개로 학습 길이를 고르고, 그 길이로 72마디 전체를 학습해 60곡을 생성합니다.
전체 학습 중에는 250 step마다 곡 4개를 끝까지 생성해, 표면의 phrase graph가 조건으로 준 유형열을 따르는지 기록합니다(감시용, 기울기 없음).

**평가 설계는 아직 정하지 않았습니다.** 아래 표의 생성 지표는 참고값입니다.

**실행 순서**: Runtime → Change runtime type에서 **GPU**를 고른 뒤 위에서 아래로 실행합니다. 안내: `docs/chart_diffusion_colab.md`.

**덮어쓰기 없음**: 결과는 `artifacts/chart_diffusion_runs/<RUN_TAG>/` 아래에만 쓰입니다.

In [ ]:
# Cell 0 — 실험 전용 clone 준비 (Colab에서만 동작합니다)
# 기존 MyDrive/DiffDodree는 건드리지 않고, 실험 branch를 별도 폴더에 clone합니다.
from pathlib import Path
import importlib.util
import os
import subprocess

REPO_URL = "https://github.com/SHlee-TDA/DiffDodree.git"
BRANCH = "claude/method-section-model-improve-d5ad4d"
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/DiffDodree_chart")

try:
    _in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _in_colab = False

if _in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    def git(*args: str) -> str:
        return subprocess.check_output(["git", "-C", str(EXPERIMENT_ROOT), *args], text=True).strip()

    if not EXPERIMENT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(EXPERIMENT_ROOT)],
            check=True,
        )
    else:
        if git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError(f"{EXPERIMENT_ROOT}는 다른 저장소의 clone입니다.")
        if git("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(
                f"{EXPERIMENT_ROOT}에 commit하지 않은 수정이 있습니다. 덮어쓰지 않도록 여기서 멈춥니다."
            )
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(EXPERIMENT_ROOT), "merge", "--ff-only", f"origin/{BRANCH}"], check=True
        )
    print("Commit:", git("log", "-1", "--oneline"))
    os.environ["GUGAKGEN_PROJECT_ROOT"] = str(EXPERIMENT_ROOT)
else:
    print("Colab이 아니므로 현재 project folder를 그대로 사용합니다.")

In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

In [ ]:
# 실험 code가 들어 있는 commit인지, GPU가 잡혔는지 확인합니다.
import subprocess

import torch

from gugakgen.workflows import chart_diffusion_experiment  # 없으면 branch가 잘못된 것입니다.

print("Commit:", subprocess.check_output(["git", "-C", str(project_root), "log", "-1", "--oneline"], text=True).strip())
print("Runtime:", runtime.as_dict())
if runtime.backend == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 연구자가 주로 바꾸는 값입니다. λ·Seed·step·격자 칸 수는 configs/chart_diffusion_v1.yaml에 있습니다.
from datetime import datetime, timezone, timedelta

RUN_MODE = "full"  # 코드 경로만 확인하려면 "smoke" (20 step, Seed 1개, 2곡)

# None이면 실행 시각으로 새 이름을 만듭니다.
# 연결이 끊겨 다시 실행할 때는 이전 실행에서 출력된 RUN_TAG를 그대로 넣으세요. 끝난 단위는 건너뜁니다.
RUN_TAG = None

CONFIG_PATH = project_root / "configs/chart_diffusion_v1.yaml"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'이어야 합니다.")
if RUN_MODE == "full" and runtime.backend != "cuda":
    raise RuntimeError("full 실행은 CUDA GPU에서만 합니다. Runtime → Change runtime type에서 GPU를 고르세요.")
if RUN_TAG is None:
    RUN_TAG = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M")
print("RUN_TAG =", repr(RUN_TAG), " ← 다시 실행할 때 이 값을 쓰세요.")

In [ ]:
# 실험 전체를 실행합니다. 학습 1회가 끝날 때마다 진행 상황을 출력합니다.
# 소요 시간은 아직 T4에서 재 보지 않았습니다. CPU에서는 그래프 손실을 켠 800 step 학습이 약 3분이었습니다.
from gugakgen.workflows.chart_diffusion_experiment import run_experiment

report = run_experiment(CONFIG_PATH, project_root, RUN_TAG, runtime, run_mode=RUN_MODE)
RUN_DIR = project_root / "artifacts/chart_diffusion_runs" / (RUN_TAG + ("" if RUN_MODE == "full" else "_smoke"))
print("저장:", RUN_DIR / "report.json")

In [ ]:
# 0) 유형화와 조건
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
print("원곡 유형열:", report["typology"]["sequence"])
print("환입(조건으로 묶은 마디):", report["returns"])
print("고른 학습 길이:", {arm: c["selected"] for arm, c in report["selected_steps"].items()})

In [ ]:
# 1) 학습 길이에 따른 보류 ELBO (칸당 nat, 낮을수록 좋음)와 감시 지표
import matplotlib.pyplot as plt

from gugakgen.workflows.chart_diffusion_experiment import curve_rows, generation_rows, monitor_rows, recall_rows

curves = pd.DataFrame(curve_rows(report))
monitor = pd.DataFrame(monitor_rows(report))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for arm, frame in curves.groupby("arm"):
    axes[0].plot(frame["step"], frame["heldout_elbo"], marker="o", label=f"{arm} · held-out")
    axes[0].plot(frame["step"], frame["train_elbo"], linestyle=":", label=f"{arm} · train")
for arm, frame in monitor.groupby("arm"):
    axes[1].plot(frame["step"], frame["first_pass_membership"], marker="o", label=f"{arm} · 유형 소속(거부 전)")
    axes[1].plot(frame["step"], frame["surface_edge_validity"], linestyle="--", label=f"{arm} · 표면 전이가 G에 있음")
    axes[2].plot(frame["step"], frame["original_phrase_rate"], marker="o", label=arm)
axes[0].set(xlabel="training step", ylabel="ELBO per cell (nats)")
axes[1].set(xlabel="training step", ylabel="share", ylim=(0, 1.02), title="생성곡의 구조 실현 (감시)")
axes[2].set(xlabel="training step", ylabel="share of phrases copied from the original", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
# 2) 암기 정도: 모두 가린 입력에서 argmax로 원곡 마디를 그대로 내는 비율
display(pd.DataFrame(recall_rows(report)).set_index("arm"))

# 3) 생성 결과 (조건마다 학습 Seed 3개 × 60곡). 참고값입니다.
display(pd.DataFrame(generation_rows(report)).set_index("arm").T)

In [ ]:
# 4) 들어 볼 악보: 조건마다 첫 학습 Seed의 첫 3곡을 MusicXML로 저장합니다(이미 있으면 멈춤).
from gugakgen.workflows.chart_diffusion_experiment import export_musicxml, load_setup

setup = load_setup(CONFIG_PATH, project_root)
listen_dir = RUN_DIR / "musicxml"
for unit in report["full"]:
    if unit["seed"] == report["full"][0]["seed"]:
        for path in export_musicxml(unit, setup, listen_dir):
            print(path.relative_to(project_root))

## 표 읽는 법

**1) 왼쪽: 보류 ELBO.** 학습에 쓰지 않은 마디를 얼마나 잘 설명하는지입니다. 내려가다 다시 오르는 지점부터 외우기 시작합니다.
전체 학습 길이는 이 곡선의 최저점으로 정했습니다.

**1) 가운데: 구조 실현(감시).** 끝까지 생성한 곡에서
- `유형 소속(거부 전)`: 각 마디가 조건으로 준 유형의 대표형에 가장 가까운 비율(편집거리). 거부로 고치기 전의 값이라 모형 자체의 능력입니다.
- `표면 전이가 G에 있음`: 음표로 다시 읽은 이웃 마디의 유형 전이가 원곡 그래프에 있는 비율. 10월 MaskGIT은 약 0.61이었습니다.

`diffusion_graph`가 `diffusion`보다 높으면 그래프 손실이 "국소 생성이 상정한 구조를 실현하게" 만든 것입니다.

**1) 오른쪽: 원곡 마디 복사율.** 높을수록 새 선율이 아니라 원곡 마디를 골라 붙인 것입니다. 선율이 새로워야 한다는 기준에서 가장 먼저 볼 값입니다.

**3) 생성 결과 (참고).**
- 구조: `first_pass_membership`(거부 전 유형 소속), `membership`(거부 후), `surface_edge_validity`, `surface_graph_tv`(표면 그래프와 조건 그래프의 총변동 거리, 0이면 같음), `returns_exact`(구성상 1).
- 새로움: `walk_positions_differing`(유형열이 원곡과 다른 자리 비율), `original_phrase_rate`, `unseen_note_transition_rate`, `note_4gram_novelty`.
- 양식(참고): `interval_jsd`, `duration_jsd`.

결과 폴더(`DiffDodree_chart/artifacts/chart_diffusion_runs/<RUN_TAG>/`)를 알려 주시면 Mac에서 읽어 해석하겠습니다.